# Preprocessing 

In [12]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("../data/loan_approval_dataset.csv")

In [5]:
df.dtypes

loan_id                       int64
no_of_dependents              int64
 education                   object
 self_employed               object
 income_annum                 int64
 loan_amount                  int64
 loan_term                    int64
 cibil_score                  int64
 residential_assets_value     int64
 commercial_assets_value      int64
 luxury_assets_value          int64
 bank_asset_value             int64
 loan_status                 object
dtype: object

In [7]:
for col in [" education", " self_employed", " loan_status"]:
    print(f"\n{col}")
    print(df[col].unique())


 education
[' Graduate' ' Not Graduate']

 self_employed
[' No' ' Yes']

 loan_status
[' Approved' ' Rejected']


In [8]:
categorical_columns = df.select_dtypes(include="object").columns

for col in categorical_columns:
    df[col] = df[col].str.strip()

In [9]:
for col in categorical_columns:
    print(f"\n{col}")
    print(df[col].unique())


 education
['Graduate' 'Not Graduate']

 self_employed
['No' 'Yes']

 loan_status
['Approved' 'Rejected']


In [13]:
numeric_columns = df.select_dtypes(include=np.number).columns

(df[numeric_columns] < 0).sum()

loan_id                       0
no_of_dependents              0
 income_annum                 0
 loan_amount                  0
 loan_term                    0
 cibil_score                  0
 residential_assets_value    28
 commercial_assets_value      0
 luxury_assets_value          0
 bank_asset_value             0
dtype: int64

In [15]:
df[df[" residential_assets_value"] < 0]


,loan_id,no_of_dependents,education,self_employed,income_annum,loan_amount,loan_term,cibil_score,residential_assets_value,commercial_assets_value,luxury_assets_value,bank_asset_value,loan_status
59,60,4,Not Graduate,Yes,5500000,18200000,16,797,-100000,4900000,18600000,4800000,Approved
196,197,4,Not Graduate,Yes,400000,1500000,2,669,-100000,600000,900000,500000,Approved
559,560,2,Graduate,Yes,200000,500000,6,885,-100000,0,300000,200000,Rejected
702,703,4,Graduate,Yes,6300000,23900000,6,899,-100000,11400000,20600000,6700000,Approved
737,738,2,Graduate,Yes,900000,2500000,16,458,-100000,100000,3200000,1100000,Rejected
784,785,0,Graduate,No,5000000,14400000,2,761,-100000,7300000,12600000,4500000,Approved
904,905,2,Graduate,No,4100000,14900000,12,571,-100000,5200000,13000000,3400000,Approved
1089,1090,3,Graduate,No,5100000,11000000,6,336,-100000,5800000,11600000,7500000,Rejected
1163,1164,2,Graduate,No,4500000,9100000,18,593,-100000,600000,12400000,2500000,Approved
1350,1351,5,Graduate,No,4000000,13700000,6,496,-100000,1400000,15800000,3700000,Rejected


In [16]:
df.loc[df[" residential_assets_value"] < 0, " residential_assets_value"].value_counts()

 residential_assets_value
-100000    28
Name: count, dtype: int64

In [17]:
df.loc[df[" residential_assets_value"] < 0, [" residential_assets_value", " loan_status"]]

,residential_assets_value,loan_status
59,-100000,Approved
196,-100000,Approved
559,-100000,Rejected
702,-100000,Approved
737,-100000,Rejected
784,-100000,Approved
904,-100000,Approved
1089,-100000,Rejected
1163,-100000,Approved
1350,-100000,Rejected


In [18]:
df[" residential_assets_value"] = df[" residential_assets_value"].replace(-100000, np.nan)

In [20]:
df[" residential_assets_value"].isnull().sum()

np.int64(28)

In [22]:
df[" cibil_score"].min(), df[" cibil_score"].max()

(300, 900)

In [23]:
df[(df[" cibil_score"] < 300) | (df[" cibil_score"] > 900)]

,loan_id,no_of_dependents,education,self_employed,income_annum,loan_amount,loan_term,cibil_score,residential_assets_value,commercial_assets_value,luxury_assets_value,bank_asset_value,loan_status


In [25]:
df[" loan_term"].unique()

array([12,  8, 20, 10,  4,  2, 18, 16, 14,  6])

In [26]:
df[" loan_term"].min(), df[" loan_term"].max()

(2, 20)

In [28]:
df["no_of_dependents"].min(), df["no_of_dependents"].max()

(0, 5)

In [29]:
df[[
    " income_annum",
    " loan_amount",
    " residential_assets_value",
    " commercial_assets_value",
    " luxury_assets_value",
    " bank_asset_value"
]].min()

income_annum                200000.0
loan_amount                 300000.0
residential_assets_value         0.0
commercial_assets_value          0.0
luxury_assets_value         300000.0
bank_asset_value                 0.0
dtype: float64

In [30]:
Q1 = df[" income_annum"].quantile(0.25)
Q3 = df[" income_annum"].quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

lower_bound, upper_bound

(np.float64(-4500000.0), np.float64(14700000.0))

In [31]:
outliers = df[
    (df[" income_annum"] < lower_bound) |
    (df[" income_annum"] > upper_bound)
]

outliers.shape

(0, 13)

In [32]:
def find_outliers_iqr(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    
    IQR = Q3 - Q1
    
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers = df[
        (df[column] < lower_bound) |
        (df[column] > upper_bound)
    ]
    
    return lower_bound, upper_bound, len(outliers)

In [33]:
financial_columns = [
    " income_annum",
    " loan_amount",
    " loan_term",
    " cibil_score",
    " residential_assets_value",
    " commercial_assets_value",
    " luxury_assets_value",
    " bank_asset_value"
]

In [34]:
for col in financial_columns:
    lower, upper, count = find_outliers_iqr(df, col)
    print(f"{col}: {count} outliers | Lower: {lower:.2f} | Upper: {upper:.2f}")

 income_annum: 0 outliers | Lower: -4500000.00 | Upper: 14700000.00
 loan_amount: 0 outliers | Lower: -13000000.00 | Upper: 42200000.00
 loan_term: 0 outliers | Lower: -9.00 | Upper: 31.00
 cibil_score: 0 outliers | Lower: 10.50 | Upper: 1190.50
 residential_assets_value: 47 outliers | Lower: -11600000.00 | Upper: 25200000.00
 commercial_assets_value: 37 outliers | Lower: -8150000.00 | Upper: 17050000.00
 luxury_assets_value: 0 outliers | Lower: -13800000.00 | Upper: 43000000.00
 bank_asset_value: 5 outliers | Lower: -4900000.00 | Upper: 14300000.00


In [35]:
res_outliers = df[
    (df[" residential_assets_value"] < -11600000) |
    (df[" residential_assets_value"] > 25200000)
]

res_outliers[" residential_assets_value"].sort_values()

2930    25300000.0
956     25300000.0
919     25300000.0
3872    25400000.0
693     25400000.0
3631    25400000.0
3157    25400000.0
2818    25500000.0
953     25500000.0
82      25500000.0
228     25500000.0
3868    25500000.0
1419    25500000.0
3498    25600000.0
262     25600000.0
1397    25700000.0
924     25800000.0
1002    25800000.0
2715    25800000.0
2185    25900000.0
98      25900000.0
2940    26100000.0
3763    26100000.0
3880    26200000.0
2412    26200000.0
4237    26200000.0
1625    26300000.0
781     26300000.0
2384    26600000.0
123     26800000.0
2828    26900000.0
1468    27000000.0
1997    27000000.0
4074    27300000.0
4042    27400000.0
3782    27500000.0
2927    27600000.0
892     27600000.0
2586    28000000.0
3234    28200000.0
987     28200000.0
714     28300000.0
1965    28400000.0
1591    28500000.0
2318    28500000.0
905     28700000.0
3119    29100000.0
Name:  residential_assets_value, dtype: float64

In [36]:
res_outliers[" residential_assets_value"].describe()

count    4.700000e+01
mean     2.654468e+07
std      1.144896e+06
min      2.530000e+07
25%      2.550000e+07
50%      2.620000e+07
75%      2.745000e+07
max      2.910000e+07
Name:  residential_assets_value, dtype: float64

In [37]:
com_outliers = df[
    (df[" commercial_assets_value"] < -8150000) |
    (df[" commercial_assets_value"] > 17050000)
]

com_outliers[" commercial_assets_value"].sort_values()

323     17200000
3541    17200000
1609    17300000
1194    17300000
1131    17300000
2349    17400000
231     17500000
2004    17500000
0       17600000
4010    17600000
3949    17600000
2933    17600000
4205    17600000
3827    17700000
2976    17700000
1812    17800000
791     17800000
4120    17900000
2302    17900000
905     17900000
1304    18200000
1029    18300000
3790    18400000
1272    18400000
3439    18400000
895     18500000
2914    18500000
3882    18500000
367     18500000
157     18700000
3000    18800000
554     18800000
1254    18900000
258     19000000
1761    19000000
2940    19000000
1768    19400000
Name:  commercial_assets_value, dtype: int64

In [38]:
com_outliers[" commercial_assets_value"].describe()

count    3.700000e+01
mean     1.807297e+07
std      6.158322e+05
min      1.720000e+07
25%      1.760000e+07
50%      1.790000e+07
75%      1.850000e+07
max      1.940000e+07
Name:  commercial_assets_value, dtype: float64

In [39]:
bank_outliers = df[
    (df[" bank_asset_value"] < -4900000) |
    (df[" bank_asset_value"] > 14300000)
]

bank_outliers[" bank_asset_value"].sort_values()

200     14400000
1633    14600000
1674    14600000
1272    14700000
1805    14700000
Name:  bank_asset_value, dtype: int64

In [40]:
bank_outliers[" bank_asset_value"].describe()

count    5.000000e+00
mean     1.460000e+07
std      1.224745e+05
min      1.440000e+07
25%      1.460000e+07
50%      1.460000e+07
75%      1.470000e+07
max      1.470000e+07
Name:  bank_asset_value, dtype: float64

In [41]:
df.isnull().sum()

loan_id                       0
no_of_dependents              0
 education                    0
 self_employed                0
 income_annum                 0
 loan_amount                  0
 loan_term                    0
 cibil_score                  0
 residential_assets_value    28
 commercial_assets_value      0
 luxury_assets_value          0
 bank_asset_value             0
 loan_status                  0
dtype: int64

In [42]:
df.duplicated().sum()

np.int64(0)

In [43]:
numeric_columns = df.select_dtypes(include=np.number).columns

(df[numeric_columns] < 0).sum()

loan_id                      0
no_of_dependents             0
 income_annum                0
 loan_amount                 0
 loan_term                   0
 cibil_score                 0
 residential_assets_value    0
 commercial_assets_value     0
 luxury_assets_value         0
 bank_asset_value            0
dtype: int64

In [45]:
for col in [" education", " self_employed", " loan_status"]:
    print(f"\n{col}")
    print(df[col].unique())


 education
['Graduate' 'Not Graduate']

 self_employed
['No' 'Yes']

 loan_status
['Approved' 'Rejected']


# TRANSFORMATION

In [46]:
df[" residential_assets_value"].isnull().sum()

np.int64(28)

In [47]:
df[df[" residential_assets_value"].isnull()][[
    " income_annum",
    " loan_amount",
    " residential_assets_value",
    " commercial_assets_value",
    " luxury_assets_value",
    " bank_asset_value",
    " loan_status"
]]

,income_annum,loan_amount,residential_assets_value,commercial_assets_value,luxury_assets_value,bank_asset_value,loan_status
59,5500000,18200000,NaN,4900000,18600000,4800000,Approved
196,400000,1500000,NaN,600000,900000,500000,Approved
559,200000,500000,NaN,0,300000,200000,Rejected
702,6300000,23900000,NaN,11400000,20600000,6700000,Approved
737,900000,2500000,NaN,100000,3200000,1100000,Rejected
784,5000000,14400000,NaN,7300000,12600000,4500000,Approved
904,4100000,14900000,NaN,5200000,13000000,3400000,Approved
1089,5100000,11000000,NaN,5800000,11600000,7500000,Rejected
1163,4500000,9100000,NaN,600000,12400000,2500000,Approved
1350,4000000,13700000,NaN,1400000,15800000,3700000,Rejected


In [48]:
df[" residential_assets_value"] = df[" residential_assets_value"].fillna(
    df[" residential_assets_value"].median()
)

# Separate Features (X) and Target (y).

In [50]:
X = df.drop(" loan_status", axis=1)
y = df[" loan_status"]

In [51]:
X.head()

,loan_id,no_of_dependents,education,self_employed,income_annum,loan_amount,loan_term,cibil_score,residential_assets_value,commercial_assets_value,luxury_assets_value,bank_asset_value
0,1,2,Graduate,No,9600000,29900000,12,778,2400000.0,17600000,22700000,8000000
1,2,0,Not Graduate,Yes,4100000,12200000,8,417,2700000.0,2200000,8800000,3300000
2,3,3,Graduate,No,9100000,29700000,20,506,7100000.0,4500000,33300000,12800000
3,4,3,Graduate,No,8200000,30700000,8,467,18200000.0,3300000,23300000,7900000
4,5,5,Not Graduate,Yes,9800000,24200000,20,382,12400000.0,8200000,29400000,5000000


In [52]:
y.head()

0    Approved
1    Rejected
2    Rejected
3    Rejected
4    Rejected
Name:  loan_status, dtype: object

# Now Train-Test Split

In [53]:
from sklearn.model_selection import train_test_split


In [54]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [55]:
X_train.shape


(3415, 12)

In [56]:
y_train.shape

(3415,)

In [57]:
X_test.shape

(854, 12)

In [58]:
y_test.shape

(854,)